# Export labeled results to Excel — gemma4:e4b + pdfplumber

The model's raw output for every document under the `gemma4-e4b_pdfplumber_whole_doc` tag lives in
`data/output/2_labeled/` as JSON: a flat array of `{"text", "label"}` items.
This notebook parses each file into a clean two-column **(label, text)** table
for review, and writes them all to one Excel workbook:

- `data/output/pdfplumber_fallback_lightonocr_and_gemma.xlsx`
- one sheet per sample, plus a `all_samples` sheet with a `sample` column

Note on the filename: the tag can include documents rescued by the LightOnOCR
fallback (a broken text layer is re-read from page images; the log records it).
Sample 11 in its current form is a clean PDF and needed no fallback.


In [1]:
import os
from pathlib import Path

if Path.cwd().name == 'notebooks':
    os.chdir(Path.cwd().parent)

import json

import pandas as pd
from IPython.display import display

TAG = 'gemma4-e4b_pdfplumber_whole_doc'
LABELED_DIR = Path('data/output/2_labeled') / TAG
XLSX = Path('data/output/pdfplumber_fallback_lightonocr_and_gemma.xlsx')

pd.set_option('display.max_colwidth', 90)


## 1. The parsing function

One function, one job: raw JSON file in, tidy `(label, text)` DataFrame out.
It validates as it parses — a malformed file or an unknown label should be
seen at review time, not silently passed through.


In [2]:
KNOWN_LABELS = ('title', 'section.title', 'section.description',
                'question.text', 'answer.text')


def parse_labeled_json(path: Path) -> pd.DataFrame:
    """Parse one raw model-output JSON into a two-column (label, text) table.

    Raises ValueError if the file is not the expected flat array of
    {'text', 'label'} objects; prints a notice for any label outside the
    five known ones rather than dropping the row.
    """
    raw = json.loads(path.read_text(encoding='utf-8'))
    if not isinstance(raw, list):
        raise ValueError(f'{path.name}: expected a JSON array, got {type(raw).__name__}')
    rows = []
    for i, item in enumerate(raw):
        if not isinstance(item, dict) or 'text' not in item or 'label' not in item:
            raise ValueError(f'{path.name}[{i}]: expected {{text, label}}, got {item!r:.60}')
        if item['label'] not in KNOWN_LABELS:
            print(f'  note: {path.name}[{i}] has unknown label {item["label"]!r}')
        rows.append({'label': item['label'], 'text': item['text'].strip()})
    return pd.DataFrame(rows, columns=['label', 'text'])


## 2. Parse every sample under the tag


In [3]:
files = sorted(LABELED_DIR.glob('sample*.json'),
               key=lambda p: int(p.stem.replace('sample', '')))
tables = {p.stem: parse_labeled_json(p) for p in files}

summary = pd.DataFrame([
    {'sample': name, 'blocks': len(df), **df['label'].value_counts().to_dict()}
    for name, df in tables.items()
]).fillna(0).astype({l: int for l in KNOWN_LABELS if any(True for _ in tables)}, errors='ignore')
display(summary.set_index('sample'))


,blocks,answer.text,question.text,section.title,title,section.description
sample,,,,,,
sample1,28,12,9,6,1,0
sample2,26,13,3,4,1,5
sample3,14,4,0,5,1,4
sample4,2,1,0,0,1,0
sample5,24,11,3,6,1,3
sample6,11,5,0,5,1,0
sample7,2,1,0,0,1,0
sample8,13,6,0,6,1,0
sample9,11,5,0,5,1,0


One table up close — the format every sheet in the workbook uses.


In [4]:
display(tables['sample1'].head(8))


,label,text
0,title,DATA MANAGEMENT AND SHARING PLAN
1,section.title,Element 1: Data Type:
2,question.text,A. Types and amount of scientific data expected to be generated in the project:
3,answer.text,"This secondary data analysis project will analyze deidentified data from 48,218 partic..."
4,question.text,"B. Scientific data that will be preserved and shared, and the rationale for doing so:"
5,answer.text,"As this is a secondary data analysis project, we will only be able to publicly share i..."
6,question.text,"C. Metadata, other relevant data, and associated documentation:"
7,answer.text,"In addition to the data described above, code and models will be included in the repos..."


## 3. Export to Excel

One sheet per sample (two columns), plus `all_samples` with a `sample` column
so the whole tag can be filtered in one view. Column widths are set so the
text is readable without resizing.


In [5]:
XLSX.parent.mkdir(parents=True, exist_ok=True)

with pd.ExcelWriter(XLSX, engine='openpyxl') as writer:
    combined = pd.concat(
        [df.assign(sample=name)[['sample', 'label', 'text']] for name, df in tables.items()],
        ignore_index=True)
    combined.to_excel(writer, sheet_name='all_samples', index=False)
    for name, df in tables.items():
        df.to_excel(writer, sheet_name=name, index=False)
    for sheet in writer.sheets.values():
        widths = {'A': 14, 'B': 22, 'C': 120} if sheet.title == 'all_samples' else {'A': 22, 'B': 130}
        for col, w in widths.items():
            sheet.column_dimensions[col].width = w

print(f'wrote {XLSX}  ({XLSX.stat().st_size / 1024:.0f} KB)')
print(f'{len(tables)} sample sheets + all_samples ({len(combined)} rows total)')


wrote data\output\pdfplumber_fallback_lightonocr_and_gemma.xlsx  (72 KB)
13 sample sheets + all_samples (193 rows total)
